In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import mne
import mtrf

If we want to study how the brain processes speech, we are faced with a fundamental limitation of traditional ERP-based approaches of analyzing EEG data: we cannot obtain average responses to repeated stimuli when presenting speech in a natural way (e.g. an audiobook). This is where linear models are useful - they allow us to estimate the relationship between the speech signal and neural response without having to epoch and average them.

The figure below illustrates the framework for modeling neural responses to naturalistic speech:
1. In the experiment, the participant listens to speech while their EEG is being recorded
2. From the speech signal, different features (envelope, spectrogram, phonemes) are extracted
3. These features are used as regressors to fit a linear model to the continuous neural recording
4. The estimated temporal response function (TRF) indicates the expected change in the neural response following a change in the predictor
5. By predicting the data, we can assess the model's accuracy

![](img/fig.webp) From [Crosse et al. (2021)](https://www.frontiersin.org/journals/neuroscience/articles/10.3389/fnins.2021.705621/full)

The model can be applied in two directions: as a **forward** (or **encoding**) model, predicting brain responses from speech features, or as a **backward** (or **decoding**) model, reconstructing stimulus features from the neural response. Forward models can be used to identify which speech features are encoded in the neural response as shown for example in [Di Liberto et al. (2015)](https://www.cell.com/current-biology/fulltext/S0960-9822(15)01001-5). Backward models can be used to measure attention in a selective listening task by testing which auditory stream can be decoded more accurately from the neural response as shown in [O'Sullivan et al. (2014)](https://academic.oup.com/cercor/article-abstract/25/7/1697/457492).

In this notebook, you are going to use the [mTRFpy](https://mtrfpy.readthedocs.io/en/latest/) package to estimate, visualize and evaluate TRF models in the context of naturalistic speech.

## Loading and Inspecting the Data

We are going to use the sample data provided by mTRFpy. This data contains a 128-channel EEG recording of a participant listening to an audiobook version of Hemingway's "The Old Man and the Sea" and the spectrogram of the corresponding audiobook segment at a sampling rate of 128 Hz. The data is split into 5 segments to allow for cross-validation when fitting the models.

In [ ]:
spectrogram, response, fs = mtrf.load_sample_data(n_segments=10)
fs

Let's plot a segment of the EEG recording. It's already preprocessed and ready for analysis.

In [ ]:
t = np.linspace(0, 1000/fs, 1000)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t, response[0][:1000], linewidth=0.4);
ax.set(xlabel="Time [s]", ylabel="Amplitude [μV]", xlim=(t.min(), t.max()));

The spectrogram contains the power of the speech signal divided into different bands across the frequency range that is relevant for human speech (80-8000 Hz).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
im = ax.imshow(spectrogram[0][:1000].T, aspect="auto", origin="lower")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Power [a.u.]")
xticks = np.linspace(0, 1000, 10).astype(int)
ax.set(xticks=xticks, xticklabels=np.round(xticks / fs, 1), xlabel="Time [s]", ylabel="Band");


We can also average the spectrogram across bands to obtain the acoustic envelope which describes the total energy in the speech signal across time.

In [ ]:
envelope = [s.mean(axis=1) for s in spectrogram]
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t, envelope[0][:1000]);
ax.set(xlabel="Time [s]", ylabel="Amplitude [a.u.]", xlim=(t.min(), t.max()));

## Encoding and Decoding Models with mTRFpy

### Background

The [mTRFpy](https://mtrfpy.readthedocs.io/en/latest/) package is a Python implementation of the Matlab [mTRF toolbox](https://github.com/mickcrosse/mTRF-Toolbox) and its purpose is to make it easy to fit, evaluate and visualize regularized linear models. The central element of the toolbox is the `TRF()` class which can be instantiated as a forward (default) or backward model (by setting `direction=-1`). This TRF can then be trained on the target stimulus and response across a range of time lags (in seconds). We can visualize the weights of the trained TRF to see the estimated temporal relationship between stimulus and response and use it to predict the response (forward modeling) or reconstruct the stimulus (backward modeling).

### Exercises

In this section you are going to apply forward and backward TRF models using the mTRFpy sample data. You are going to train the models, visualize their weights and predict the original responses or stimuli while playing with the different TRF parameters to get a better understanding of the interface. Here are the essential code snippets:


| Description | Code |
|---|---|
| Create a forward TRF model | `trf = mtrf.TRF()` |
| Create a backward TRF model | `trf = mtrf.TRF(direction=-1)` |
| Train the TRF | `trf.train(stimulus, response, fs, tmin=0, tmax=0.4, regularization=100)` |
| Predict the response and compute the score | `pred, score = trf.predict(stimulus, response)` |
| Plot the TRF | `trf.plot()` |
| Plot the TRF for a specific channel | `trf.plot(channel=105)` |
| Plot the global field power (GFP) as an image | `trf.plot(channel="gfp", kind="image")` |
| Convert a backward to a forward model | `trf_fwd = trf.to_forward(response)` |

**Example**: Train a TRF to the `response` using the acoustic `envelope` with time lags between `tmin=0` and `tmax=0.3` with $\lambda=1$ (`regularization`).

In [ ]:
trf = mtrf.TRF()
trf.train(envelope, response, fs, tmin=0, tmax=0.3, regularization=1)
trf.plot();

**Exercise**: Fit the same model again with $\lambda=100$ and plot the results.

In [ ]:
trf = mtrf.TRF()
trf.train(envelope, response, fs, tmin=0, tmax=0.3, regularization=100)
trf.plot();

**Exercise**: Fit the same model again but increase the number of time lags to span the range from `tmin=-0.1` to `tmax=0.5` and plot the result.

In [ ]:
trf = mtrf.TRF()
trf.train(envelope, response, fs, tmin=-0.1, tmax=0.5, regularization=100)
trf.plot();

**Example**: Use the trained TRF to predict the response from the `envelope` and plot a segment of the prediction and original response for channel 1.

**Note**: The second output of `trf.predict` is the prediction score which we assign to `_` because we do not need it here.

In [ ]:
response_pred, _ = trf.predict(envelope, response)
ch_idx = 1
plt.plot(response[0][:500, ch_idx], label="Original")
plt.plot(response_pred[0][:500, ch_idx], label="Prediction")
plt.legend()

**Exercise**: The cell below fits the same model but with $\lambda=0.1$. Predict using the `envelope` and plot a segment of the predicted and original response. How did reducing $\lambda$ affect the model's prediction?

In [ ]:
trf = mtrf.TRF()
trf.train(envelope, response, fs, tmin=-0.1, tmax=0.5, regularization=0.1)

In [ ]:
response_pred, _ = trf.predict(envelope, response)
ch_idx = 1
plt.plot(response[0][:500, ch_idx], label="Original")
plt.plot(response_pred[0][:500, ch_idx], label="Prediction")
plt.legend();

**Exercise**: The code below fits a backward model with `direction=-1` that reconstructs the envelope from the neural response. Use `trf.predict` with `response=response` and plot a segment of the original envelope together with the reconstruction.

**Note**: Fitting this model takes longer because there are 128 predictors (i.e. EEG channels) now.

In [ ]:
trf = mtrf.TRF(direction=-1)
trf.train(envelope, response, fs, tmin=0, tmax=0.4, regularization=10)

In [ ]:
envelope_pred, _ = trf.predict(envelope, response)
plt.plot(envelope[0][:500], label="Original")
plt.plot(envelope_pred[0][:500], label="Prediction")
plt.legend();

**Exercise**: Plot the TRF from the previous exercise. What does the warning say? Why are the time lags negative?

In [ ]:
trf.plot();

**Exercise**: The cell below transforms the backward to a forward model with interpretable weights (for details see [Haufe et al. 2014](https://www.sciencedirect.com/science/article/pii/S1053811913010914)). Plot the weights of `fwd_trf`.

In [ ]:
trf_fwd = trf.to_forward(response)

In [ ]:
trf_fwd.plot();

**Exercise**: Fit a forward TRF to predict the `response` from the `spectrogram` with `tmin=0`, `tmax=0.4` and `regularization=100`. Plot it with `channel=105` to visualize the TRF at this (left auditory) channel for every spectral band.

In [ ]:
trf = mtrf.TRF()
trf.train(spectrogram, response, fs, tmin=0, tmax=0.4, regularization=1000)
trf.plot(channel=105);

**Exercise**: Plot the TRF again with `channel="gfp"` to plot the global field power (i.e. the standard deviation across channels).

In [ ]:
trf.plot(channel="gfp");

**Exercise**: Plot the TRF again with `channel="gfp"` and `kind="image"` to plot the global field power as an image which allows you to visually separate the frequency bands.

In [ ]:
trf.plot(channel="gfp", kind="image");

## Optimizing Models and Conrolliung for Overfitting

In [ ]:
trf = mtrf.TRF()
trf.train(envelope, response, fs, tmin=0, tmax=0.3, regularization=10)

In [ ]:
montage = mne.channels.make_standard_montage("biosemi128")
evoked = trf.to_mne_evoked(montage)[0]

In [ ]:
evoked.info["ch_names"][106]

In [ ]:
evoked.plot_sensors(show_names=True)
evoked.info["ch_names"]

In [ ]:
evoked.plot_joint();

In [ ]:
pred, score = trf.predict(envelope, response, average=False)
mne.viz.plot_topomap(score, evoked.info, size=4)

In [ ]:
trf = mtrf.TRF(direction=-1)
trf.train(envelope, response, fs, tmin=0, tmax=0.3, regularization=1000)

In [ ]:
trf_fwd = trf.to_forward(response)
evoked = trf_fwd.to_mne_evoked(montage)[0]
evoked.plot_joint();

In [ ]:
pred, scores = trf.predict(envelope, response, average=False)

In [ ]:
mtrf.stats.crossval(trf, envelope, response, fs, tmin=0.3, tmax=0.3, regularization=10)

## Statistical Inference

In [ ]:
trf = mtrf.TRF()
score = mtrf.stats.crossval(trf, envelope, response, fs, tmin=0, tmax=0.5, regularization=2)
score

In [ ]:
scores_permute = mtrf.stats.permutation_distribution(
    trf, envelope, response, fs, tmin=0, tmax=0.5, regularization=10, n_permute=200
    )

In [ ]:
plt.hist(scores_permute);
plt.axvline(score)

In [ ]:
(scores_permute>score).sum()/len(scores_permute)

## Comparing TRFs to ERPs